# Tesis - MDM UBA - 2026

**Tariff classification using NLP**

By Santiago Tedoldi

# Fine-tuned model viz

In [1]:
import os
import json
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Model classes and embedding function

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel

class TokenizedDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }
    
class HSClassifier(nn.Module):
    def __init__(self,
                 n_classes: int,
                 fine_tune: bool = False,
                 n_finetune_layers: int = 0):
        """
        Args:
          n_classes:      number of target classes
          fine_tune:      if True, you’ll unfreeze either all or the last layers
          n_finetune_layers:
                          • =0 (default) → if fine_tune=True, unfreeze *all* DistilBERT layers  
                          • >0             → unfreeze only that many of the *last* transformer blocks  
                          • ignored if fine_tune=False (encoder stays fully frozen)
        """
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # Freeze everything by default
        for param in self.distilbert.parameters():
            param.requires_grad = False

        # If fine_tune, decide what to unfreeze
        if fine_tune:
            if n_finetune_layers > 0:
                # Unfreeze only the last `n_finetune_layers` transformer blocks
                for block in self.distilbert.transformer.layer[-n_finetune_layers:]:
                    for param in block.parameters():
                        param.requires_grad = True
            else:
                # n_finetune_layers == 0 → unfreeze *all* DistilBERT params
                for param in self.distilbert.parameters():
                    param.requires_grad = True

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.distilbert.config.hidden_size, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.3),
            nn.Linear(1024, n_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # Take <CLS> token representation
        logits = self.classifier(hidden_state)
        return logits
    
class HSDataset(Dataset):
    def __init__(self, dataframe, tokenizer, desc_col='', label_col='', max_length=500):
        self.descriptions = dataframe[desc_col].tolist()
        self.labels = dataframe[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.descriptions[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False,
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': self.labels[idx],
            'description': self.descriptions[idx]
        }
        return item
    
# Function to extract [CLS] embeddings
def get_embeddings(dataloader, model, device):
    all_embeds = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attn = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attn)
            # DistilBERT does not have pooler; use first token hidden state
            cls_embeds = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeds.append(cls_embeds)
            all_labels.extend(batch['label'])
    return np.vstack(all_embeds), np.array(all_labels)
    

Loading finetuned model

In [3]:
# reading labels
with open('models/labels_dict.json', 'r') as f:
    labels_dict = json.load(f)

label2id = labels_dict['label2id']
id2label = labels_dict['id2label']

# loading tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# loading model
model_name = 'distiltbert_hs04_simple_classifier_finetuned_3epochs_13062025_150328.pth'

model_save_path = os.path.join('models', model_name)

model_finetuned = HSClassifier(n_classes=len(label2id), fine_tune=True)

model_finetuned.load_state_dict(torch.load(model_save_path))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model_finetuned.to(device)

model_finetuned.eval() # to set the model for inference

C:\Users\santt\AppData\Local\Temp\ipykernel_36976\1868374554.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_finetuned.load_state_dict(torch.load(model_save_path)

Using device: cuda


HSClassifier(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Lin

Getting embeddings from full finetune test

In [4]:
# Reading the dataset
results_finetuned = pd.read_csv('results/hs04_distiltbert_finetuned_results.csv', index_col=0)

# Getting chapter labels for viz
results_finetuned['True Chapter'] = results_finetuned['True Label'].astype(str).str[:2]

# Build dataset and loader - sampled for viz comparison with EDA
ds = HSDataset(results_finetuned.sample(len(results_finetuned)//2), 
               tokenizer, desc_col='Description', 
               label_col='True Chapter', max_length=300)

loader = DataLoader(ds, batch_size=32, shuffle=False)

embeds_finetuned, labels_finetuned = get_embeddings(loader, model_finetuned.distilbert, device)
print("Embeddings shape:", embeds_finetuned.shape)

Embeddings shape: (13389, 768)


In [5]:
labels_finetuned

array(['84', '82', '87', ..., '94', '87', '84'], dtype='<U2')

#### Viz with PCA and TSNE

In [6]:
from pca_tsne_viz_plotly import pca_viz_embs, tsne_viz_embs
from plotly.colors import qualitative as qual

PLOTLY_PALETTE = (
    qual.Plotly + qual.D3 + qual.Set1 + qual.Set2 + qual.Set3 + qual.Dark24 + qual.Light24
)

def _sorted_unique_labels(labels):
    return sorted(np.unique(labels).tolist())

def _label2color(labels):
    uniq = _sorted_unique_labels(labels)
    return {lab: PLOTLY_PALETTE[i % len(PLOTLY_PALETTE)] for i, lab in enumerate(uniq)}

df_hs06 = pd.read_csv('data/hs06_full_eng.csv', index_col='hs06', 
                      dtype={'hs06': str, 'full_eng': str},
                      usecols=['hs06', 'full_eng'])
df_hs06['HS04'] = df_hs06.index.str[:4]
df_hs06['HS02'] = df_hs06.index.str[:2]

colors_all = _label2color(df_hs06["HS02"].unique())

In [7]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df_raw = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

df_raw['HS04'] = df_raw['HS06'].str[:4]
df_raw['HS02'] = df_raw['HS06'].str[:2]

df_raw.drop_duplicates(inplace=True)

def freq_table(df, col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    return summary

# Chapter-level (HS02)
hs02_freq = freq_table(df_raw, 'HS02', 'chapter')

Viz embeddings with PCA

In [8]:
title = "Goods Description TOP 10 sampled - 2D PCA of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_goods_desc_finetuned_pca_top10_2d.html"

topN_labels = hs02_freq.head(10).index.to_list()

fig_pca_2d, _, pca_desc_2d = pca_viz_embs(
    embeds_finetuned, labels_finetuned,
    n_componets=2,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=topN_labels,
    label2color=colors_all
)

fig_pca_2d.show()

📄 Saved plot to emb_goods_desc_finetuned_pca_top10_2d.html


In [9]:
title = "Goods Description TOP 10 sampled - 3D PCA of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_goods_desc_finetuned_pca_top10_3d.html"

topN_labels = hs02_freq.head(10).index.to_list()

fig_pca_3d, _, pca_desc_3d = pca_viz_embs(
    embeds_finetuned, labels_finetuned,
    n_componets=3,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=topN_labels,
    label2color=colors_all
)

fig_pca_3d.show()

📄 Saved plot to emb_goods_desc_finetuned_pca_top10_3d.html


Viz embeddings with TSNE

In [10]:
title = "Goods Description TOP 10 sampled - 2D t-SNE of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_goods_desc_finetuned_tsne_top10_2d.html"

topN_labels = hs02_freq.head(10).index.to_list()

fig_tsne_2d, _, tsne_desc_2d = tsne_viz_embs(
    embeds_finetuned, labels_finetuned,
    n_componets=2,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=topN_labels,
    label2color=colors_all,
    init='pca',
    perplexity=30)

fig_tsne_2d.show()

📄 Saved plot to emb_goods_desc_finetuned_tsne_top10_2d.html


In [11]:
title = "Goods Description TOP 10 sampled - 3D t-SNE of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_goods_desc_finetuned_tsne_top10_3d.html"

topN_labels = hs02_freq.head(10).index.to_list()

fig_tsne_3d, _, tsne_desc_3d = tsne_viz_embs(
    embeds_finetuned, labels_finetuned,
    n_componets=3,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=topN_labels,
    label2color=colors_all,
    init='pca',
    perplexity=30)

fig_tsne_3d.show()

📄 Saved plot to emb_goods_desc_finetuned_tsne_top10_3d.html


Viz embs from full_eng nomenclature

In [12]:
# Build dataset and loader - sampled for viz comparison with EDA
ds = HSDataset(df_hs06, 
               tokenizer, desc_col='full_eng', 
               label_col='HS02', max_length=300)

loader = DataLoader(ds, batch_size=32, shuffle=False)

embeds_full_eng, labels_full_eng = get_embeddings(loader, model_finetuned.distilbert, device)
print("Embeddings shape:", embeds_full_eng.shape)

Embeddings shape: (6064, 768)


In [13]:
labels_full_eng

array(['01', '01', '01', ..., '96', '97', '97'], dtype='<U2')

Viz embeddings with PCA

Using nomenclature based PCA

In [14]:
title = "HS06 English Nomenclature ALL - 2D PCA of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_hs06_nomen_finetuned_pca_nomen_all_2d.html"

fig_pca_2d, _, pca_nomen_2d = pca_viz_embs(
    embeds_full_eng, labels_full_eng,
    n_componets=2,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=None,
    label2color=colors_all
)

fig_pca_2d.show()

📄 Saved plot to emb_hs06_nomen_finetuned_pca_nomen_all_2d.html


In [15]:
title = "HS06 English Nomenclature ALL - 3D PCA of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_hs06_nomen_finetuned_pca_nomen_all_3d.html"

fig_pca_3d, _, pca_nomen_3d = pca_viz_embs(
    embeds_full_eng, labels_full_eng,
    n_componets=3,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=None,
    label2color=colors_all
)

fig_pca_3d.show()

📄 Saved plot to emb_hs06_nomen_finetuned_pca_nomen_all_3d.html


Viz embeddings with TSNE

In [16]:
title = "HS06 English Nomenclature ALL - 2D t-SNE of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_hs06_nomen_finetuned_tsne_nomen_all_2d.html"

fig_tsne_2d, _, tsne_nomen_2d = tsne_viz_embs(
    embeds_full_eng, labels_full_eng,
    n_componets=2,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=None,
    label2color=colors_all,
    init='pca',
    perplexity=30)

fig_tsne_2d.show()

📄 Saved plot to emb_hs06_nomen_finetuned_tsne_nomen_all_2d.html


In [ ]:
title = "HS06 English Nomenclature ALL - 3D t-SNE of DistilBERT Finetuned"
legend_title="HS02 Chapter"
html_output_file="emb_hs06_nomen_finetuned_tsne_nomen_all_3d.html"

fig_tsne_3d, _, tsne_nomen_3d = tsne_viz_embs(
    embeds_full_eng, labels_full_eng,
    n_componets=3,
    title=title,
    legend_title=legend_title,
    html_output_file=html_output_file,
    filter_labels=None,
    label2color=colors_all,
    init='pca',
    perplexity=30)

fig_tsne_3d.show()

📄 Saved plot to emb_hs06_nomen_finetuned_tsne_nomen_all_3d.html
